In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from tqdm import tqdm
from shapely import box
from shapely.ops import transform
import pyproj
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

from sklearn.cluster import KMeans

basin_file = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs/metadata/All_MERIT_matchups.gpkg")
subbasins = gpd.read_file(basin_file).set_index('comid')

In [2]:
from data import BasinDeltaTable
save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")
root_dir = save_dir / 'deltalakes' / 'training'
store = BasinDeltaTable(root_dir)

In [57]:
processed_basins = store.get_processing_status(source='era5')
processed_basins

,basin,subbasin,source,processed_at,has_data
0,ABOM-108118010,56127328,era5,2025-09-18 19:37:11.684490+00:00,True
1,ABOM-108118010,56127842,era5,2025-09-18 19:37:11.684490+00:00,True
2,ABOM-108118010,56127858,era5,2025-09-18 19:37:11.684490+00:00,True
3,ABOM-108118010,56127880,era5,2025-09-18 19:37:11.684490+00:00,True
4,ABOM-108118010,56128692,era5,2025-09-18 19:37:11.684490+00:00,True
...,...,...,...,...,...
240,ABOM-100288010,ABOM-102903010,era5,2025-09-18 19:12:09.361867+00:00,True
241,ABOM-100288010,ABOM-102984010,era5,2025-09-18 19:12:09.361867+00:00,True
242,ABOM-100288010,ABOM-97444010,era5,2025-09-18 19:12:09.361867+00:00,True
243,ABOM-107434010,ABOM-107434010,era5,2025-09-18 19:02:46.841634+00:00,True


In [58]:
processed_basins['basin'].unique()

array(['ABOM-108118010', 'ABOM-11748010', 'ABOM-117649010',
       'ABOM-117568010', 'ABOM-117109010', 'ABOM-113129010',
       'ABOM-113192010', 'ABOM-117406010', 'ABOM-100288010',
       'ABOM-107434010'], dtype=object)

In [18]:
e5_dir = Path("/nas/cee-water/cjgleason/data/ERA5-Land/")

def create_spatial_batches(gdf, batch_size):
    """
    Groups subbasins into spatially coherent batches using K-Means clustering.
    
    Returns a list of batch dictionaries, each containing the IDs and
    a single bounding box for all geometries in that batch.
    """
    n_batches = int(np.ceil(len(gdf)/batch_size))
        
    # Get centroids for clustering
    centroids = np.array([list(p.coords)[0] for p in gdf.geometry.centroid])
    
    # Perform K-Means clustering
    kmeans = KMeans(n_clusters=n_batches, random_state=42, n_init=10)
    gdf['cluster'] = kmeans.fit_predict(centroids)
    
    batches = []
    for i in range(n_batches):
        batch_gdf = gdf[gdf['cluster'] == i]
        if batch_gdf.empty:
            continue
        
        # Get all COMIDs in this cluster
        batch_ids = batch_gdf.index.tolist()
        
        # Calculate a single, unified bounding box for the entire batch
        # Add a 0.1 degree padding to ensure all grid cells are included
        min_lon, min_lat, max_lon, max_lat = batch_gdf.total_bounds
        padding = 0.1
        batch_bounds = (
            np.floor((min_lon - padding) * 10) / 10,
            np.floor((min_lat - padding) * 10) / 10,
            np.ceil((max_lon + padding) * 10) / 10,
            np.ceil((max_lat + padding) * 10) / 10,
        )
        
        batches.append({
            "ids": batch_ids,
            "geoms": batch_gdf.geometry,
            "bounds": batch_bounds
        })
        
    return batches


def open_monthly_files(year, month):
    """Loads and merges the original monthly ERA5 files."""
    fps = [
        e5_dir / "accumulated" / f"{year}_{month:02d}.nc",
        e5_dir / "average" / f"{year}_{month:02d}.nc"
    ]
    datasets = [xr.open_dataset(fp) for fp in fps]
    ds = xr.merge(datasets).drop_vars(['expver','number'], errors='ignore')
    return ds


def subset_ds_by_bounds(ds, bounds):
    """Subset dataset by spatial bounds."""
    min_lon, min_lat, max_lon, max_lat = bounds
    return ds.sel(latitude=slice(max_lat, min_lat), longitude=slice(min_lon, max_lon))


def get_weight_matrix(ds_subset, watershed_geom):
    """Calculate weight matrix for a single watershed."""
    lat, lon = ds_subset['latitude'].values, ds_subset['longitude'].values
    lat_res, lon_res = 0.1, 0.1
    
    watershed_proj = watershed_geom.to_crs(epsg=3857)
    transformer = pyproj.Transformer.from_crs("epsg:4326", "epsg:3857", always_xy=True)
    
    coverage_fractions = np.zeros((len(lat), len(lon)))
    
    for i, lat_val in enumerate(lat):
        for j, lon_val in enumerate(lon):
            pixel_box = box(lon_val - lon_res/2, lat_val - lat_res/2,
                            lon_val + lon_res/2, lat_val + lat_res/2)
            pixel_box_proj = transform(transformer.transform, pixel_box)
            
            intersection_area = watershed_proj.intersection(pixel_box_proj).area
            pixel_area = pixel_box_proj.area
            coverage_fractions[i, j] = (intersection_area / pixel_area).iloc[0] if pixel_area > 0 else 0
    
    return xr.DataArray(
        coverage_fractions,
        dims=["latitude", "longitude"],
        coords={"latitude": lat, "longitude": lon}
    )


def get_results_df(ds_sub, weights):
    """Calculate weighted statistics."""
    dims = ["latitude", "longitude"]
    ds_sub_w = ds_sub.weighted(weights)
    
    mean_df = (ds_sub_w.mean(dim=dims).to_dataframe()
               .rename(lambda n: n+"_mean", axis=1))
    var_df = (ds_sub_w.var(dim=dims).to_dataframe()
               .rename(lambda n: n+"_var", axis=1))
    
    return pd.concat([mean_df, var_df], axis=1)


def process_spatial_batch(spatial_batch: dict, start_date, end_date):
    """
    Processes a spatially coherent batch of COMIDs.
    Loads regional data once per month and applies it to all COMIDs in the batch.
    """
    batch_ids = spatial_batch['ids']
    batch_geoms = gpd.GeoDataFrame(spatial_batch['geoms'])
    batch_bounds = spatial_batch['bounds']

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    
    # Pre-calculate weight matrices for all watersheds in the batch
    # This requires a sample dataset to get the grid right
    print(f"Pre-calculating weight matrices for batch...")
    sample_ds = open_monthly_files(start_date.year, start_date.month)
    sample_ds_sub = subset_ds_by_bounds(sample_ds, batch_bounds)
    
    weights_map = {}
    for comid, row in batch_geoms.iterrows():
        watershed = gpd.GeoDataFrame([row], crs=batch_geoms.crs)
        weights_map[comid] = get_weight_matrix(sample_ds_sub, watershed)
    sample_ds.close()
    print(f"Weight matrices calculated.")

    # Generate date range
    date_range = pd.date_range(start=start_date, end=end_date, freq="MS")
    
    # This dictionary will hold the time-series data for each COMID
    comid_results = {comid: [] for comid in batch_ids}

    # Outer loop is TIME
    for date in date_range:
        # Load the full monthly file and subset to the BATCH's bounds ONCE
        ds_full = open_monthly_files(date.year, date.month)
        ds_region = subset_ds_by_bounds(ds_full, batch_bounds)
        
        # Inner loop is COMID (processing from in-memory data)
        for comid in batch_ids:
            weights = weights_map[comid]
            
            # Ensure weights align with the regional data subset
            weights_aligned = weights.reindex_like(ds_region, method='nearest', tolerance=0.01)
            
            df_month = get_results_df(ds_region, weights_aligned)
            comid_results[comid].append(df_month)
        
        ds_full.close()
        ds_region.close()

    cat_dfs = {}
    for comid, df_list in comid_results.items():
        cat_dfs[comid] = pd.concat(df_list, axis=0)
    return cat_dfs

In [4]:
processed_basins = store.get_processing_status(source='era5')
to_process = subbasins[~subbasins.index.isin(processed_basins['subbasin'])]
to_process

,outlet,outlet_id,total_area,unitarea,reservoir,custom,reach_id,sword_area,sword_distance,lake_reach_ids,...,longitude,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider,hybas_area_diff,geometry
comid,,,,,,,,,,,,,,,,,,,,,
21000001,POINT (5.889166666666659 47.94916666666667),EAUF-V7200010,398.5,152.9,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,-0.034201,"MULTIPOLYGON (((5.93875 47.99125, 5.93708 47.9..."
21000012,POINT (5.889166666666659 47.94916666666667),EAUF-V7200010,324.8,194.6,False,False,NaN,NaN,NaN,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.053341,"MULTIPOLYGON (((5.79042 48.01125, 5.79042 48.0..."
21000019,POINT (5.684999999999993 47.53416666666667),EAUF-V7200010,5173.0,242.5,False,False,2.160280e+10,4530.071328,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,-0.007977,"MULTIPOLYGON (((5.76042 47.56125, 5.76042 47.5..."
21000021,POINT (5.76916666666666 47.58083333333334),EAUF-V7200010,4758.7,8.4,False,False,2.160280e+10,4278.612301,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.003021,"MULTIPOLYGON (((5.80625 47.60125, 5.80708 47.6..."
21000022,POINT (5.80416666666666 47.57),EAUF-V7200010,4506.6,68.9,False,False,2.160280e+10,4278.612301,0.0,[],...,NaN,NaT,NaT,NaN,NaN,NaN,NaN,None,0.009101,"MULTIPOLYGON (((5.85875 47.57458, 5.85875 47.5..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-410401112134801,POINT (-112.1875 41.0625),USGS-410401112134801,9900.3,296.5,False,True,NaN,NaN,NaN,[],...,-112.230256,2003-10-01,2025-09-08,0.006230,45.873291,11.263905,7165.0,usgs,-0.002093,"MULTIPOLYGON (((-112.21458 40.96042, -112.2154..."
USGS-50035000,POINT (-66.4592 18.3217),USGS-50037000,345.7,345.7,False,True,NaN,NaN,NaN,[],...,-66.459568,1950-01-01,2025-09-08,0.240693,1857.585136,7.406389,25691.0,usgs,-0.091982,"MULTIPOLYGON (((-66.49208 18.29625, -66.49208 ..."
USGS-50037000,POINT (-66.5 18.3983),USGS-50037000,429.1,83.4,False,True,NaN,NaN,NaN,[],...,-66.496560,2019-06-13,2025-09-08,1.265763,911.802460,11.017469,2247.0,usgs,0.167805,"MULTIPOLYGON (((-66.50125 18.33125, -66.50125 ..."


In [5]:
for basin_name, basin_gdf in to_process.groupby('outlet_id'):
    print(f"Starting basin {basin_name}")
    batches = create_spatial_batches(basin_gdf, 16)
    print(f"  Split basin into {len(batches)} batches")
    for batch in tqdm(batches):
        results = process_spatial_batch(
            batch,
            '2020-01-01',
            '2020-02-01'
        )
        break
    break
    

Starting basin ABOM-100288010


/tmp/ipykernel_2785269/3625038556.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = np.array([list(p.coords)[0] for p in gdf.geometry.centroid])


  Split basin into 1 batches


  0%|          | 0/1 [00:00<?, ?it/s]

Pre-calculating weight matrices for batch...
Weight matrices calculated.


  0%|          | 0/1 [00:07<?, ?it/s]


In [49]:
len(all_batches)

2693

In [ ]:
all_batches = []
for basin_name, basin_gdf in tqdm(to_process.groupby('outlet_id')):
    batches = create_spatial_batches(basin_gdf, 16)
    for b in batches:
        all_batches.append((basin_name, b))
    # if len(batches)>1:
    #     break

    # if len(all_batches)>30:
    #     break


In [55]:
from dask.distributed import Client, as_completed
from dask_jobqueue import SLURMCluster

ImportError: dask.distributed is not installed.

Please either conda or pip install distributed:

  conda install dask distributed             # either conda install
  python -m pip install "dask[distributed]" --upgrade    # or pip install

In [47]:
all_batches[0][1]

{'ids': ['56150341',
  '56150344',
  '56150871',
  '56150898',
  '56150932',
  '56151111',
  '56151157',
  '56151379',
  '56151534',
  '56154788',
  '56156020',
  'ABOM-100288010',
  'ABOM-101896010',
  'ABOM-102903010',
  'ABOM-102984010',
  'ABOM-97444010'],
 'geoms': comid
 56150341          MULTIPOLYGON (((116.25208 -32.91875, 116.25125...
 56150344          MULTIPOLYGON (((116.42708 -32.95875, 116.42708...
 56150871          MULTIPOLYGON (((116.42875 -32.84208, 116.42875...
 56150898          MULTIPOLYGON (((116.70958 -32.75292, 116.70875...
 56150932          MULTIPOLYGON (((116.86875 -32.76375, 116.86875...
 56151111          MULTIPOLYGON (((116.77375 -32.86375, 116.76792...
 56151157          MULTIPOLYGON (((117.08292 -32.65458, 117.08292...
 56151379          MULTIPOLYGON (((117.37625 -32.83542, 117.37625...
 56151534          MULTIPOLYGON (((116.55542 -32.77875, 116.55542...
 56154788          MULTIPOLYGON (((116.82958 -33.17042, 116.82958...
 56156020          MULTIPOLYGON (

In [6]:
results['ABOM-97444010']

,ssrd_mean,strd_mean,ro_mean,sro_mean,e_mean,tp_mean,t2m_mean,d2m_mean,fal_mean,sp_mean,ssrd_var,strd_var,ro_var,sro_var,e_var,tp_var,t2m_var,d2m_var,fal_var,sp_var
valid_time,,,,,,,,,,,,,,,,,,,,
2020-01-01,3.324392e+07,2.776505e+07,0.000008,0.000000e+00,-0.001159,8.523463e-07,296.115914,283.759036,0.113176,98003.682494,1.525451e+09,4.933858e+09,3.543706e-11,0.000000e+00,2.445995e-08,0.000000e+00,0.116333,0.145891,0.000101,81566.262367
2020-01-02,3.258692e+07,2.889416e+07,0.000008,0.000000e+00,-0.001140,2.139807e-06,291.028574,283.205026,0.113304,98130.688434,4.920413e+08,9.245106e+09,3.413993e-11,0.000000e+00,2.152948e-08,0.000000e+00,0.096500,0.099020,0.000101,83618.010790
2020-01-03,2.563287e+07,2.880397e+07,0.000009,3.577655e-07,-0.000869,2.026716e-04,288.246249,275.635446,0.113430,98656.428943,5.860112e+10,6.141630e+08,3.245300e-11,1.473872e-14,1.458645e-08,2.681553e-09,0.098350,0.053819,0.000101,85000.874600
2020-01-04,2.360289e+07,2.598892e+07,0.000008,1.533841e-07,-0.000729,8.561527e-05,291.487310,273.894622,0.113558,98624.069062,3.995086e+11,1.999513e+10,3.185935e-11,1.554197e-14,1.386029e-08,4.482660e-09,0.026604,0.243668,0.000101,82347.700579
2020-01-05,3.375689e+07,2.547441e+07,0.000008,0.000000e+00,-0.001022,2.581446e-06,297.071659,278.967316,0.113683,98127.863193,8.692825e+08,7.005869e+09,3.136960e-11,0.000000e+00,2.145372e-08,0.000000e+00,0.052715,0.149700,0.000101,78556.163108
2020-01-06,3.298480e+07,2.856133e+07,0.000008,0.000000e+00,-0.001020,1.718958e-06,298.308401,281.026954,0.113809,97844.779376,5.450582e+08,5.712399e+09,3.031153e-11,0.000000e+00,1.394777e-08,4.484155e-44,0.048787,0.191553,0.000101,77849.325547
2020-01-07,3.245600e+07,2.953366e+07,0.000008,0.000000e+00,-0.001077,8.523463e-07,297.199307,282.611985,0.113934,97481.731439,8.822800e+08,5.377294e+09,2.928866e-11,0.000000e+00,1.458151e-08,0.000000e+00,0.136803,0.247598,0.000101,78266.501854
2020-01-08,3.253307e+07,2.902613e+07,0.000008,0.000000e+00,-0.000998,0.000000e+00,294.797053,284.488663,0.114062,97630.743227,3.201206e+09,4.577394e+09,2.820583e-11,0.000000e+00,1.372042e-08,0.000000e+00,0.096539,0.061125,0.000101,80304.088680
2020-01-09,3.241272e+07,2.845800e+07,0.000008,0.000000e+00,-0.000958,0.000000e+00,293.093542,280.104191,0.114188,98246.689782,9.968543e+08,9.507617e+09,2.724158e-11,0.000000e+00,1.976459e-08,0.000000e+00,0.176508,0.012512,0.000101,81586.847085


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 6))

for i, batch in enumerate(batches):
    xmin, ymin, xmax, ymax = batch['batch_bounds']
    width = xmax - xmin
    height = ymax - ymin

    rect = patches.Rectangle(
        (xmin, ymin), width, height,
        linewidth=1.5, edgecolor='C0',
        facecolor='C0', alpha=0.3, label=f"Batch {i+1}" if i == 0 else None
    )
    ax.add_patch(rect)

# Make it look nice
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Batch Bounding Boxes")
ax.autoscale()
ax.legend()

plt.show()